<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/genneral_sankey_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note before using Plotly in Jupiter lab it is necessary not only to install the Pyton libary but also the Jupiter lab extension
for instance micromamba install -c conda-forge jupyterlab-plotly-extension

# Sankey opsætning

##Indlæs libraries

In [15]:
import requests
import json
import pandas as pd
import plotly.graph_objects as go
import duckdb

## Create blueprint for creating objects


In [16]:
class Table:
    """
    Represents a database table and handles fetching of data from Airtable,
    storing it in a pandas DataFrame. It manages label and relationship
    lists with lazy loading and includes Airtable's primary key for each record.

    Attributes:
        table_id (str): Identifier for the table.
        label_column (str): Label of the table to be used in diagrams.
        api_key (str): API key for accessing Airtable.
        base_id (str): Base ID of the Airtable database.
        foreign_key_table_id (str): Identifier of the table referenced in the foreign_key_column.
        foreign_key_column (str): Column name that acts as a foreign key to another table.
        _data (DataFrame): Internal DataFrame containing fetched data.
        _labels (list): List of tuples containing record IDs and labels, lazily loaded.
        _relationships (list): List of source-target tuples based on foreign keys, lazily loaded.
    """

    def __init__(self, table_id, label_column, api_key, base_id, foreign_key_table_id, foreign_key_column, show=True):
        self.table_id = table_id
        self.label_column = label_column
        self.api_key = api_key
        self.base_id = base_id
        self.foreign_key_table_id = foreign_key_table_id
        self.foreign_key_column = foreign_key_column
        self._data = None
        self._labels = None
        self._relationships = None

    @property
    def data(self):
        if self._data is None:
            self.fetch_data()
        return self._data

    def fetch_data(self):
        """Fetches and populates the internal DataFrame with primary key and record fields."""
        url = f"https://api.airtable.com/v0/{self.base_id}/{self.table_id}"
        headers = {"Authorization": f"Bearer {self.api_key}"}
        params = {}
        data = []

        while True:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code != 200:
                raise Exception(f"Failed to fetch data: {response.text}")
            page_data = response.json()
            for record in page_data['records']:
                record_data = record['fields']
                record_data['id'] = record['id']  # Include the primary key
                data.append(record_data)

            if 'offset' in page_data:
                params['offset'] = page_data['offset']
            else:
                break

        self._data = pd.DataFrame(data)

    @property
    def labels(self):
        if self._labels is None:
            self.create_label_and_relationship_lists()
        return self._labels

    @property
    def relationships(self):
        if self._relationships is None:
            self.create_label_and_relationship_lists()
        return self._relationships

    def create_label_and_relationship_lists(self):
        """Generates labels and source-target relationships from data."""
        if self._data is None:
            self.fetch_data()

        self._labels = [(self.table_id+"_"+ row['id'], row[self.label_column]) for index, row in self._data.iterrows()]
        if self.foreign_key_column != "":
            df_relationship = self._data.explode(self.foreign_key_column)
            self._relationships = [(row['id'], row[self.foreign_key_column]) for index, row in df_relationship.iterrows() if self.foreign_key_column in row]
        else:
            self._relationships = []

# Example usage (make sure the field names are correct for your Airtable setup)
# tables = [
#     Table(table_id="tblmO1yIO7iLGjeBx", label_column="Name", api_key=api_key, base_id=base_id, foreign_key_table_id="tblO8e0GuUpzcnCOh", foreign_key_column="Phenomenon"),
#     Table(table_id="tblO8e0GuUpzcnCOh", label_column="Name", api_key=api_key, base_id=base_id, foreign_key_table_id="tblWUnluzfa79Y26z", foreign_key_column="Variable"),
#     Table(table_id="tblWUnluzfa79Y26z", label_column="Name", api_key=api_key, base_id=base_id, foreign_key_table_id="", foreign_key_column=""),
# ]



## Opsætning af Sankey-diagram

In [3]:
def create_sankey_diagram(tables, title="Sankey Diagram"):
    # Filtrer tabeller, så dem med show=False ikke vises i diagrammet
    visible_tables = [t for t in tables if getattr(t, 'show', True)]

    # Maps to store indices of each label in all tables
    label_to_index = {}
    current_index = 0

    # Lists for Sankey diagram
    node_labels = []
    source_indices = []
    target_indices = []
    values = []

    # First Phase: Index all labels from all tables
    for table in visible_tables:
        for id_with_table, actual_label in table.labels:
            if id_with_table not in label_to_index:
                label_to_index[id_with_table] = current_index
                node_labels.append(actual_label)  # Append actual label for visualization
                current_index += 1

    # Second Phase: Process relationships now that all labels are indexed
    for table in visible_tables:
        for source_id, target_id in table.relationships:
            # Create full unique IDs for source and target using the correct table IDs
            source_full_id = f"{table.table_id}_{source_id}"
            target_full_id = f"{table.foreign_key_table_id}_{target_id}"

            if source_full_id in label_to_index and target_full_id in label_to_index:
                source_index = label_to_index[source_full_id]
                target_index = label_to_index[target_full_id]
                source_indices.append(source_index)
                target_indices.append(target_index)
                values.append(1)  # Value can be adjusted if needed

    # Create the Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
        ),
        link=dict(
            source=source_indices,
            target=target_indices,
            value=values
        ))])

    fig.update_layout(
        font_size=10,
        autosize=True,
        width= 1920,
        height=1080,
        margin=dict(
            l=10,
            r=10,
            b=10,
            t=20,
            pad=4,

        ),
        title_text=title,
        paper_bgcolor="white"
    )
    fig.show()

# Example usage
# Assuming 'tables' is a list of Table instances that have already fetched data and generated labels and relationships
# create_sankey_diagram(tables)


# Diagram 2
Target Group ->Targets (keyword title) - > land uses

Target group kan evt kategoriseres som 1, 2, 3 med angivelse under figuren

Keyword title er under udarbejdelse


In [4]:
# Load data from airtable
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'
tables = [

    Table(table_id="tblVarbVYd96JUE6f", label_column="Target Group", api_key=api_key, base_id=base_id, foreign_key_table_id="fldju2kIp9TjBKsSo", foreign_key_column="Targets"),
    Table(table_id="tbl7OYOXduME11uh7", label_column="Target name", api_key=api_key, base_id=base_id, foreign_key_table_id="fldmVdtx08Ic59sj6",foreign_key_column="Land uses"),
    Table(table_id="tblTRyuT48bBN24QG", label_column="Name", api_key=api_key, base_id=base_id, foreign_key_table_id="", foreign_key_column=""),
]


# Filtrér den første tabel: Group
df_group = tables[0].data
df_group_filtered = df_group[df_group["Target Group"].notnull() & df_group["Target Group"].astype(str).str.strip().ne("")]
tables[0]._data = df_group_filtered

# Filtrér den anden tabel: Targets
df_targets = tables[1].data
df_targets_filtered = df_targets[df_targets["Target name"].notnull() & df_targets["Target name"].astype(str).str.strip().ne("")]
tables[1]._data = df_targets_filtered


Test

In [5]:
print(tables[0].labels)

[('tblVarbVYd96JUE6f_rec2F1hkw7WEobaaO', 'Klimasikring'), ('tblVarbVYd96JUE6f_rec6quEtcMxawpJkL', 'Friluftsinteresser'), ('tblVarbVYd96JUE6f_recCpKBy5x0lPgtvg', 'Nye former for organisering '), ('tblVarbVYd96JUE6f_recDviCE0bwaajN1u', 'Divers landbrugssektor'), ('tblVarbVYd96JUE6f_recE0Qu6qHUDNg3Ko', 'Fødevaresikkerhed'), ('tblVarbVYd96JUE6f_recHfjiv83M1OVdvd', 'Økonomi og regional udvikling'), ('tblVarbVYd96JUE6f_recLHHNJWEFiwEGZ6', 'Forestry and Biomass'), ('tblVarbVYd96JUE6f_recM72plqmOMokKav', 'Sårbar natur'), ('tblVarbVYd96JUE6f_recMyT5YOWmU5Bs5j', 'Kulturarv'), ('tblVarbVYd96JUE6f_recO2namYwsWydIP5', 'Særligt hensyn til truede og naturligt indvandrende arter'), ('tblVarbVYd96JUE6f_recQAGiyZOcHgzbN7', 'Kvælstoffikserende afgrøder'), ('tblVarbVYd96JUE6f_recQLsdhedkibpIKg', 'Biodiversity'), ('tblVarbVYd96JUE6f_recQfAWZsDtcEXTXD', 'Sundhed'), ('tblVarbVYd96JUE6f_recRZNnC7bJjfYu2u', 'Naturlig vanddynamik'), ('tblVarbVYd96JUE6f_recSipJAiWSy8WqOL', 'Fisk og fiskerierhverv'), ('tblVarbVYd

#Construct diagram 2

In [14]:
# Construct the diagram
label_to_index = {}
node_labels = []
source_indices = []
target_indices = []
values = []
create_sankey_diagram(tables)